# 05 — Multi-objective ranking → ranked rescue / maintenance / remission candidates

> ⚠️ **Not medical advice.** Research/hypothesis use only.

What this notebook does:
1. Joins `MCAS_Compound_Library_v1.csv` with `outputs/qsar_predictions.csv` (from notebook 02), any `outputs/docking_*.csv` files (from notebook 04), and `outputs/reinvent_generated.csv` (from notebook 03).
2. Computes a weighted multi-objective score per compound:
   - QSAR safety (lower hERG / AMES / DILI = better)
   - Docking strength against the relevant target for each category
   - Evidence level boost (high > medium > low)
3. Writes ranked CSVs per category to `outputs/ranked_<category>.csv`.
4. Optionally updates the top of each `hypotheses/<category>.md` with the new top-10.

In [ ]:
from pathlib import Path
import pandas as pd
import glob

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT_DIR = REPO_ROOT / 'outputs'
library = pd.read_csv(REPO_ROOT / 'data' / 'compounds' / 'MCAS_Compound_Library_v1.csv')
print('library rows:', len(library))

In [ ]:
qsar_path = OUT_DIR / 'qsar_predictions.csv'
if qsar_path.exists():
    qsar = pd.read_csv(qsar_path)
    df = library.merge(qsar.drop(columns=['category', 'subcategory', 'evidence_level'], errors='ignore'), on='name', how='left')
else:
    df = library.copy()
    print('No QSAR predictions yet — run notebook 02 first.')
df.head()

In [ ]:
for path in sorted(glob.glob(str(OUT_DIR / 'docking_*.csv'))):
    target = Path(path).stem.replace('docking_', '')
    dock = pd.read_csv(path)
    df = df.merge(
        dock.rename(columns={'score': f'dock_{target}'})[['name', f'dock_{target}']],
        on='name', how='left'
    )
df.head()

In [ ]:
import numpy as np

EVIDENCE_WEIGHT = {'high': 1.0, 'medium': 0.6, 'low': 0.3, '': 0.0}

# Per-category preferred targets for docking weight
CATEGORY_TARGETS = {
    'rescue':      ['HRH1', 'HRH2', 'CYSLTR1'],
    'maintenance': ['CYSLTR1', 'HRH1', 'BTK'],
    'remission':   ['MRGPRX2', 'KIT', 'NFE2L2', 'GLP1R'],
}

def composite_score(row):
    s = EVIDENCE_WEIGHT.get(row.get('evidence_level', ''), 0.0)
    if 'hERG_score' in row and pd.notna(row['hERG_score']):
        s += (1.0 - row['hERG_score']) * 0.3  # lower hERG = better
    for t in CATEGORY_TARGETS.get(row.get('category', ''), []):
        col = f'dock_{t}'
        if col in row and pd.notna(row[col]):
            s += (-row[col]) * 0.2  # smina/Vina: more-negative = better binding
    return s

df['composite_score'] = df.apply(composite_score, axis=1)
df.sort_values('composite_score', ascending=False).head(20)

In [ ]:
for category in ['rescue', 'maintenance', 'remission']:
    sub = df[df['category'] == category].sort_values('composite_score', ascending=False)
    out = OUT_DIR / f'ranked_{category}.csv'
    sub.to_csv(out, index=False)
    print(f'{category}: {len(sub)} -> {out.name}')
print('Done. Top hits per category written to outputs/.')

## Next

- Inspect `outputs/ranked_*.csv` and copy the top 10 into the corresponding `hypotheses/<category>.md` 'Top AI-ranked candidates' section.
- For any novel REINVENT-generated SMILES that score high, request synthesis via Enamine REAL Space (`mcule.com`, `enamine.net`) and propose wet-lab validation (LAD2 β-hexosaminidase / CD63 / tryptase ELISA — see `docs/wet_lab_protocols.md`).
- Open a PR with updated hypothesis docs and the relevant ranked CSVs.